In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn import feature_extraction, linear_model, model_selection, preprocessing

In [2]:
train_df = pd.read_csv("train.csv")

In [3]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7613 entries, 0 to 7612
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   id        7613 non-null   int64
 1   keyword   7552 non-null   str  
 2   location  5080 non-null   str  
 3   text      7613 non-null   str  
 4   target    7613 non-null   int64
dtypes: int64(2), str(3)
memory usage: 297.5 KB


In [4]:
import sys
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn import svm
from xgboost import XGBClassifier

documents = train_df["text"]

labels = train_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    documents, labels, test_size=0.25, random_state=42
)

tfidf = TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=(1, 2))

X_train_matrix = tfidf.fit_transform(X_train)
X_test_matrix = tfidf.transform(X_test)

text_clf = MultinomialNB()
text_svm = svm.SVC(kernel='linear')
text_xgb = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1)

text_clf.fit(X_train_matrix, y_train)
text_svm.fit(X_train_matrix, y_train)
text_xgb.fit(X_train_matrix, y_train)

y_pred_clf = text_clf.predict(X_test_matrix)
y_pred_svm = text_svm.predict(X_test_matrix)
y_pred_xgb = text_xgb.predict(X_test_matrix)


print("Accuracy CLF:", accuracy_score(y_test, y_pred_clf))
print("\nClassification Report CLF:\n", classification_report(y_test, y_pred_clf))

print("Accuracy svm:", accuracy_score(y_test, y_pred_svm))
print("\nClassification Report svm:\n", classification_report(y_test, y_pred_svm))

print("Accuracy xgb:", accuracy_score(y_test, y_pred_xgb))
print("\nClassification Report xgb:\n", classification_report(y_test, y_pred_xgb))

Accuracy CLF: 0.7977941176470589

Classification Report CLF:
               precision    recall  f1-score   support

           0       0.78      0.91      0.84      1091
           1       0.84      0.65      0.73       813

    accuracy                           0.80      1904
   macro avg       0.81      0.78      0.78      1904
weighted avg       0.80      0.80      0.79      1904

Accuracy svm: 0.7951680672268907

Classification Report svm:
               precision    recall  f1-score   support

           0       0.82      0.83      0.82      1091
           1       0.76      0.75      0.76       813

    accuracy                           0.80      1904
   macro avg       0.79      0.79      0.79      1904
weighted avg       0.79      0.80      0.80      1904

Accuracy xgb: 0.7326680672268907

Classification Report xgb:
               precision    recall  f1-score   support

           0       0.70      0.94      0.80      1091
           1       0.85      0.45      0.59       8

In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {
            "n_estimators": [50, 100],
            "max_depth": [3, 4, 5],
            "learning_rate": [0.01, 0.1, 0.2],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0]
        }

search = GridSearchCV(XGBClassifier(), param_grid, refit=True, cv=3, verbose=2)

search.fit(X_train_matrix, y_train)

Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.8; total time=   0.8s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.8; total time=   0.7s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.8; total time=   0.8s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=50, subsample=1.0; total time=   1.3s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=50, subsample=1.0; total time=   4.0s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=50, subsample=1.0; total time=   3.0s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   5.4s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   5.1s
[CV] END colsamp

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.01, 0.1, ...], 'max_depth': [3, 4, ...], 'n_estimators': [50, 100], ...}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_

In [6]:
print("Best Parameters:", search.best_params_)
print("Best Estimator:", search.best_estimator_)
best_model_xgb = search.best_estimator_

y_pred = best_model_xgb.predict(X_test_matrix)
print(classification_report(y_test, y_pred))

Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100, 'subsample': 0.8}
Best Estimator: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)
              precision    recall  f1-score   support

           0       0.73      0.92      0.82      1091
 

In [7]:
param_grid = {
'C': [0.1, 1, 10],
'gamma': [1, 0.1, 0.01],
'kernel': ['linear', 'rbf', 'poly']
}

search = GridSearchCV(svm.SVC(), param_grid, refit=True, cv=3, verbose=2)

search.fit(X_train_matrix, y_train)

Fitting 3 folds for each of 27 candidates, totalling 81 fits


[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   8.5s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   7.4s
[CV] END ......................C=0.1, gamma=1, kernel=linear; total time=   7.4s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   8.1s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   8.6s
[CV] END .........................C=0.1, gamma=1, kernel=rbf; total time=   9.3s
[CV] END ........................C=0.1, gamma=1, kernel=poly; total time=   7.7s
[CV] END ........................C=0.1, gamma=1, kernel=poly; total time=   7.9s
[CV] END ........................C=0.1, gamma=1, kernel=poly; total time=   8.3s
[CV] END ....................C=0.1, gamma=0.1, kernel=linear; total time=   8.1s
[CV] END ....................C=0.1, gamma=0.1, kernel=linear; total time=   7.9s
[CV] END ....................C=0.1, gamma=0.1, kernel=linear; total time=   7.7s
[CV] END ...................

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 1, ...], 'gamma': [1, 0.1, ...], 'kernel': ['linear', 'rbf', ...]}"
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a

In [8]:
print("Best Parameters:", search.best_params_)
print("Best Estimator:", search.best_estimator_)
best_model_svm = search.best_estimator_

y_pred = best_model_svm.predict(X_test_matrix)
print(classification_report(y_test, y_pred))

Best Parameters: {'C': 1, 'gamma': 1, 'kernel': 'linear'}
Best Estimator: SVC(C=1, gamma=1, kernel='linear')
              precision    recall  f1-score   support

           0       0.82      0.83      0.82      1091
           1       0.76      0.75      0.76       813

    accuracy                           0.80      1904
   macro avg       0.79      0.79      0.79      1904
weighted avg       0.79      0.80      0.80      1904

